# NEST point-neuron simulation of a simplified circuit

This notebook builds a `NestCircuitSimulationScanConfig` with the blocks the NEST backend supports,
generates the SONATA simulation from it, and runs that simulation with obi-one's NEST runner,
`run_simulation`, the function the `circuit_simulation_nest_machine` task calls on the platform.

The circuit is a circuit simplified by sonata-simplify and exported for NEST, in the
SONATA-extension format: its neurons are a `nest_point` population, and its projections are
`virtual` populations, which stay silent unless spikes are replayed from them.

What gets exercised:

| Group | Blocks |
| --- | --- |
| Neuron sets | `PointPopulationIDNeuronSet`, `VirtualPopulationNeuronSet` |
| Timestamps | `SingleTimestamp` |
| Stimuli | `PoissonSpikeStimulus`, `FullySynchronousSpikeStimulus`, `ConstantCurrentClampSomaticStimulus`, `LinearCurrentClampSomaticStimulus`, `MultiPulseCurrentClampSomaticStimulus`, `SimulationDtSinusoidalCurrentClampSomaticStimulus` |
| Recordings | `SomaVoltageRecording`, `TimeWindowSomaVoltageRecording`, `SimulationDtTimeWindowSomaVoltageRecording` |
| Synaptic manipulations | `DisconnectSynapticManipulation`, `ConnectSynapticManipulation`, `ScaleWeightSynapticManipulation` |

**Prerequisites:**
- NEST in the notebook's environment: `uv sync --extra nest` (or `pip install obi-one[nest]`).
- A NEST circuit. By default, the newest NEST export written by
  [run_simplification_local.ipynb](../../circuit_simplification/run_simplification_local.ipynb),
  whose default algorithm, `adex_nest`, simplifies the tiny nbS1 circuit in `examples/data`. Set
  `OBI_NEST_CIRCUIT` to the `circuit_config.json` of another NEST circuit to use that one instead.

**Writes to:** a fresh run directory under `obi-output/nest_simulations/` outside the repository.
Nothing is registered in the database.

## 1. Imports

In [ ]:
import json
import os
import tempfile
from pathlib import Path

import libsonata
import matplotlib.pyplot as plt
import numpy as np

from obi_one.core.exception import ConfigValidationError
from obi_one.core.info import Info
from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one.core.tuple import NamedTuple
from obi_one.scientific.blocks.neuron_sets.id import PointPopulationIDNeuronSet
from obi_one.scientific.blocks.neuron_sets.population import VirtualPopulationNeuronSet
from obi_one.scientific.blocks.recordings.soma import (
    SimulationDtTimeWindowSomaVoltageRecording,
    SomaVoltageRecording,
    TimeWindowSomaVoltageRecording,
)
from obi_one.scientific.blocks.stimuli.spike.fully_synchronous import FullySynchronousSpikeStimulus
from obi_one.scientific.blocks.stimuli.spike.poisson import PoissonSpikeStimulus
from obi_one.scientific.blocks.stimuli.stimulus import (
    ConstantCurrentClampSomaticStimulus,
    LinearCurrentClampSomaticStimulus,
    MultiPulseCurrentClampSomaticStimulus,
    SimulationDtSinusoidalCurrentClampSomaticStimulus,
)
from obi_one.scientific.blocks.synaptic_manipulations.connect_disconnect import (
    ConnectSynapticManipulation,
    DisconnectSynapticManipulation,
    ScaleWeightSynapticManipulation,
)
from obi_one.scientific.blocks.timestamps.single import SingleTimestamp
from obi_one.scientific.library.circuit import Circuit
from obi_one.scientific.tasks.generate_simulations.config.nest.nest_circuit import (
    NestCircuitSimulationScanConfig,
)

start = globals().get("repo_root", Path.cwd())
repo_root = next(
    p for p in (start, *start.parents)
    if (p / "pyproject.toml").is_file() and (p / "examples").is_dir()
)

## 2. The circuit

The NEST config accepts `nest_point` and `virtual` node populations only, and needs at least one
`nest_point` population. The projection used below is `VPM` when the circuit has one (the tiny
nbS1 circuit has `VPM` and `POm`), and its first virtual population otherwise.

In [ ]:
exports = sorted(
    (repo_root.parent / "obi-output/circuit_simplification").glob(
        "run_*/0/*/output_nest_*/circuit_config.json"
    ),
    key=lambda path: path.stat().st_mtime,
)
circuit_config = os.environ.get("OBI_NEST_CIRCUIT") or (exports[-1] if exports else None)
if circuit_config is None:
    raise FileNotFoundError(
        "No NEST circuit found. Run run_simplification_local.ipynb with a NEST algorithm "
        "(adex_nest by default), or set OBI_NEST_CIRCUIT to a NEST circuit_config.json."
    )
circuit = Circuit(name="simplified_nest_circuit", path=str(Path(circuit_config).resolve()))

nodes = circuit.sonata_circuit.nodes
print(f"Circuit: {circuit.path}")
for name in nodes.population_names:
    print(f"  {name:32} {nodes[name].type:12} {nodes[name].size:5} nodes")

POINT_POPULATION = next(
    name for name in nodes.population_names if nodes[name].type == "nest_point"
)
virtual_populations = [name for name in nodes.population_names if nodes[name].type == "virtual"]
PROJECTION = "VPM" if "VPM" in virtual_populations else virtual_populations[0]
print(f"\nSimulated: {POINT_POPULATION}; projection used below: {PROJECTION}")

## 3. Neuron sets and timestamps

Point and virtual neuron sets are offered. `Probe` is the three point neurons the current clamps
inject into and most recordings record from; `Projection` is one projection population.

Every untargeted block falls back to `Default: All Point Neurons`, except the source of a spike
stimulus: when every virtual population projects onto point neurons, as in a simplified circuit,
it falls back to `Default: All Virtual Neurons`, so an untargeted spike stimulus drives the
circuit through its projections.

In [ ]:
SIM_DURATION = 1000.0  # ms

sim_conf = NestCircuitSimulationScanConfig.empty_config()
sim_conf.set(
    Info(
        campaign_name="NEST feature tour",
        campaign_description="NEST-supported blocks on a simplified circuit",
    ),
    name="info",
)

probe = PointPopulationIDNeuronSet(
    population=POINT_POPULATION, neuron_ids=NamedTuple(name="probe", elements=[0, 1, 2])
)
sim_conf.add(probe, name="Probe")

projection = VirtualPopulationNeuronSet(population=PROJECTION)
sim_conf.add(projection, name="Projection")

cut = SingleTimestamp(start_time=300.0)
sim_conf.add(cut, name="Cut")
restore = SingleTimestamp(start_time=600.0)
sim_conf.add(restore, name="Restore")

## 4. Stimuli

The NEST runner understands four SONATA input modules: `synapse_replay` for the spike stimuli,
and `linear`, `pulse` and `sinusoidal` for the current clamps. Amplitudes are in nanoamps.

A stimulus without timestamps starts at 0 ms plus its `timestamp_offset`, which places the four
current clamps one after the other here.

In [ ]:
# `synapse_replay`: untargeted, so 5 Hz Poisson trains of every virtual neuron, replayed through
# the projections onto every point neuron, for the whole run.
sim_conf.add(PoissonSpikeStimulus(frequency=5.0, duration=SIM_DURATION), name="Background")

# `synapse_replay`: one synchronous volley of the whole projection at 800 ms.
sim_conf.add(
    FullySynchronousSpikeStimulus(source_neuron_set=projection.ref, timestamp_offset=800.0),
    name="Volley",
)

# `linear`: a step, then a ramp.
sim_conf.add(
    ConstantCurrentClampSomaticStimulus(
        neuron_set=probe.ref, timestamp_offset=100.0, duration=50.0, amplitude=0.5
    ),
    name="Step",
)
sim_conf.add(
    LinearCurrentClampSomaticStimulus(
        neuron_set=probe.ref,
        timestamp_offset=180.0,
        duration=100.0,
        amplitude_start=0.0,
        amplitude_end=0.8,
    ),
    name="Ramp",
)

# `pulse`: a 50 Hz train of 5 ms pulses.
sim_conf.add(
    MultiPulseCurrentClampSomaticStimulus(
        neuron_set=probe.ref,
        timestamp_offset=650.0,
        duration=100.0,
        amplitude=0.8,
        width=5.0,
        frequency=50.0,
    ),
    name="Pulses",
)

# `sinusoidal`: sampled on the simulation timestep, hence the `SimulationDt` prefix.
sim_conf.add(
    SimulationDtSinusoidalCurrentClampSomaticStimulus(
        neuron_set=probe.ref,
        timestamp_offset=880.0,
        duration=100.0,
        maximum_amplitude=0.4,
        frequency=20.0,
    ),
    name="Sine",
)

## 5. Recordings

Only soma voltage is reported. A NEST recording keeps its own Timestep, which must be a multiple
of the 0.025 ms simulation timestep; the `SimulationDt` blocks record on the simulation timestep
itself. The recordings without a neuron set record every point neuron.

NEST holds every sample until the run ends, so generation refuses recordings that add up to more
than 40 million samples (neurons x frames, summed over the recordings).

In [ ]:
sim_conf.add(SomaVoltageRecording(neuron_set=probe.ref, dt=0.1), name="ProbeVoltage")
sim_conf.add(
    TimeWindowSomaVoltageRecording(dt=1.0, start_time=250.0, end_time=650.0),
    name="AllVoltageAroundCut",
)
sim_conf.add(
    SimulationDtTimeWindowSomaVoltageRecording(
        neuron_set=probe.ref, start_time=795.0, end_time=830.0
    ),
    name="ProbeVolley",
)

## 6. Synaptic manipulations

`Connect`, `Disconnect` and `ScaleWeight` become SONATA `connection_overrides`. The NEST runner
applies each one from its time on, which must fall on a simulation step. An override's `weight`
is a factor on the circuit weight, so the `Connect` at 600 ms restores the projection's synapses
to their circuit weight. The untargeted `ScaleWeight` applies to the synapses between point
neurons, the default pre- and postsynaptic neuron set.

In [ ]:
sim_conf.add(
    DisconnectSynapticManipulation(presynaptic_neuron_set=projection.ref, timestamps=cut.ref),
    name="CutProjection",
)
sim_conf.add(
    ConnectSynapticManipulation(presynaptic_neuron_set=projection.ref, timestamps=restore.ref),
    name="RestoreProjection",
)
sim_conf.add(ScaleWeightSynapticManipulation(weight_scaling=2.0), name="StrongerRecurrence")

## 7. Initialize

The simulation runs on all point neurons, and NEST ignores the initial voltage of SONATA
`conditions`, so neither is set here.

In [ ]:
sim_conf.set(
    NestCircuitSimulationScanConfig.Initialize(circuit=circuit, simulation_length=SIM_DURATION),
    name="initialize",
)

validated_sim_conf = sim_conf.validated_config()

## 8. Generate the simulation

This writes `simulation_config.json`, `node_sets.json` and the spike files of the two spike
stimuli. Generation also refuses, before anything is written, what the NEST runner would refuse
once launched; see section 10.

In [ ]:
output_base = repo_root.parent / "obi-output/nest_simulations"
output_base.mkdir(parents=True, exist_ok=True)
output_root = Path(tempfile.mkdtemp(prefix="run_", dir=output_base))

grid_scan = GridScanGenerationTask(
    form=validated_sim_conf,
    output_root=str(output_root),
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()
run_tasks_for_generated_scan(grid_scan)

coordinate_dir = output_root / "0"
simulation_config_path = coordinate_dir / "simulation_config.json"
print(simulation_config_path)

## 9. What was generated

In [ ]:
sonata_config = json.loads(simulation_config_path.read_text())

print("target_simulator:", sonata_config["target_simulator"])
print("run:", sonata_config["run"])
print()
print("inputs:")
for name, entry in sonata_config["inputs"].items():
    print(f"  {name:12} {entry['module']:15} -> {entry['node_set']:28} "
          f"from {entry['delay']} ms for {entry['duration']} ms")
print()
print("reports:")
for name, entry in sonata_config["reports"].items():
    print(f"  {name:20} {entry['cells']:28} dt={entry['dt']} "
          f"window=({entry['start_time']}, {entry['end_time']})")
print()
print("connection_overrides:")
for entry in sonata_config["connection_overrides"]:
    print(f"  {entry['name']:20} {entry['source']} -> {entry['target']} "
          f"weight={entry['weight']} from t={entry['delay']} ms")

`Background` replays from `Default: All Virtual Neurons`: its spike file holds the spikes of
every projection population. Each report keeps its own `dt` and window.

## 10. Generation refuses what NEST would refuse

For example, a recording Timestep the runner cannot sample: 0.03 ms is not a multiple of the
0.025 ms simulation timestep. The same holds for a manipulation timed between two steps, a pulse
longer than its period, a current clamp on no `nest_point` neuron, and recordings over the memory
limit.

In [ ]:
refused = NestCircuitSimulationScanConfig.empty_config()
refused.set(Info(campaign_name="Refused", campaign_description="Refused"), name="info")
refused.add(SomaVoltageRecording(dt=0.03), name="TooFine")
refused.set(
    NestCircuitSimulationScanConfig.Initialize(circuit=circuit, simulation_length=SIM_DURATION),
    name="initialize",
)

refused_scan = GridScanGenerationTask(
    form=refused.validated_config(),
    output_root=str(output_root / "refused"),
    coordinate_directory_option="ZERO_INDEX",
)
refused_scan.execute()
try:
    run_tasks_for_generated_scan(refused_scan)
except ConfigValidationError as error:
    print("Refused:", error)

## 11. Run the simulation

`run_simulation` builds the network with PyNEST, runs it, and writes the spike report and the
voltage reports into the simulation's output directory. It is imported here, since it needs NEST;
everything above only needs obi-one.

In [ ]:
from obi_one.scientific.library.simulation.nest.simulate_nest import run_simulation

run_simulation(simulation_config_path, num_threads=1)
output_dir = coordinate_dir / sonata_config["output"]["output_dir"]
print(sorted(path.name for path in output_dir.iterdir()))

## 12. Results

The spike report and the voltage reports are SONATA files, read here with libsonata.

In [ ]:
spikes = libsonata.SpikeReader(str(output_dir / sonata_config["output"]["spikes_file"]))
spike_nodes, spike_times = (
    np.array(values) for values in zip(*spikes[POINT_POPULATION].get(), strict=True)
)

windows = {"0-300 ms": (0, 300), "300-600 ms (projection cut)": (300, 600), "600-1000 ms": (600, 1000)}
for label, (begin, end) in windows.items():
    print(f"{label:30} {int(((spike_times >= begin) & (spike_times < end)).sum()):4} spikes")


def soma_report(name):
    frame = libsonata.SomaReportReader(str(output_dir / f"{name}.h5"))[POINT_POPULATION].get()
    return np.asarray(frame.ids), np.asarray(frame.times), np.asarray(frame.data)


for name in sonata_config["reports"]:
    ids, times, data = soma_report(name)
    print(f"{name:20} {len(ids)} neurons x {len(times)} frames, dt {times[1] - times[0]:.3f} ms")

In [ ]:
clamps = {"Step": (100, 150), "Ramp": (180, 280), "Pulses": (650, 750), "Sine": (880, 980)}
fig, (raster, trace, volley) = plt.subplots(3, 1, figsize=(11, 9))

raster.scatter(spike_times, spike_nodes, s=8, color="#1f77b4")
raster.axvspan(300, 600, color="grey", alpha=0.2, label=f"{PROJECTION} disconnected")
raster.axvline(800, color="#d62728", lw=0.8, label=f"{PROJECTION} volley")
raster.set_xlim(0, SIM_DURATION)
raster.set_ylabel("node id")
raster.set_title(f"{POINT_POPULATION}: {len(spike_times)} spikes")
raster.legend(loc="upper right", fontsize="small")

ids, times, data = soma_report("ProbeVoltage")
for column, node in enumerate(ids):
    trace.plot(times, data[:, column], lw=0.7, label=f"neuron {node}")
for label, (begin, end) in clamps.items():
    trace.axvspan(begin, end, color="orange", alpha=0.15)
    trace.text((begin + end) / 2, 1.02, label, ha="center", transform=trace.get_xaxis_transform())
trace.set_xlim(0, SIM_DURATION)
trace.set_ylabel("V (mV)")
trace.set_title("Probe soma voltage, 0.1 ms Timestep (current clamps shaded)", pad=16)
trace.legend(loc="lower right", fontsize="small", ncol=3)

ids, times, data = soma_report("ProbeVolley")
for column, node in enumerate(ids):
    volley.plot(times, data[:, column], lw=0.8, label=f"neuron {node}")
volley.axvline(800, color="#d62728", lw=0.8)
volley.set_xlabel("time (ms)")
volley.set_ylabel("V (mV)")
volley.set_title(f"The {PROJECTION} volley at 800 ms, on the 0.025 ms simulation timestep")

fig.tight_layout()

The raster shows the replay driving the circuit through its projections, the firing that drops
while the projection is disconnected from 300 to 600 ms, and the response to the synchronous
volley at 800 ms. The probe traces show the four current clamps.

On the platform, the same config generated for a circuit registered in the database (referenced
with `CircuitFromID`, as in the Brian2 example) is run by the `circuit_simulation_nest_machine`
task, which calls the same `run_simulation`.